# Current Price Prediction From `product_snapshots`

This notebook builds a regression workflow to predict `current_price` from the local SQLite database at `instance/pricing_prediction.db`.

## Current run summary

- Dataset used: `product_snapshots` joined with `products` and `product_images`
- Rows with target: `13,587`
- Distinct `sku_id`: `12,811`
- Image rows available: `48,855`
- Queries covered: `6`
- Primary metric: `RMSE`
- Secondary metrics: `MAE`, `R2`
- Structured validation: `GroupKFold(n_splits=5)` grouped by `sku_id`
- Image validation: `KFold(n_splits=5, shuffle=True, random_state=42)` on a unique-SKU image sample

## Recorded leaderboard from the initial run

| experiment | family | scope | RMSE mean | RMSE std | MAE mean | R2 mean |
|---|---|---|---:|---:|---:|---:|
| `cb_listing_plus_price_position` | catboost | full dataset | 45.8882 | 3.7036 | 17.7949 | 0.8460 |
| `cb_core_plus_listing` | catboost | full dataset | 48.2305 | 3.7295 | 18.8418 | 0.8300 |
| `cb_core_structured` | catboost | full dataset | 64.8424 | 5.0117 | 33.7213 | 0.6930 |
| `cb_subset_structured_plus_image` | catboost + pytorch image embeddings | image sample of 360 SKUs | 78.9462 | 20.4869 | 49.4321 | 0.4379 |
| `cb_subset_structured` | catboost | image sample of 360 SKUs | 79.3127 | 22.1919 | 48.1199 | 0.4317 |

## Winner model

```text
Winner model: cb_listing_plus_price_position
Family: catboost
Primary metric: rmse = 45.8882
Secondary metrics: mae = 17.7949, r2 = 0.8460
CV detail: 41.8493, 43.1609, 43.8732, 51.3321, 49.2254
Key features: query, brand, seller, rating/review features, title-derived features, original_price, discount_pct, ranking position, image_count/image namespace
Key params: depth=8, learning_rate=0.05, iterations=450, log1p target
Notebook: notebook/20260312-current-price-product-snapshots.ipynb
Next iteration: run a text-aware CatBoost model with `title` as a text feature and a richer hybrid image experiment on a larger unique-SKU sample
```


## Environment and package note

The project environment does not include the data science stack by default. The initial run used ephemeral dependencies through `uv run --with ...`.

Useful package set for reproducing this notebook:

```bash
uv run   --with numpy   --with pandas   --with scikit-learn   --with catboost   --with torch   --with torchvision   --with pillow   --with httpx   jupyter lab
```


In [ ]:
from __future__ import annotations

import io
import math
import re
import sqlite3
from concurrent.futures import ThreadPoolExecutor, as_completed
from pathlib import Path

import httpx
import numpy as np
import pandas as pd
import torch
from PIL import Image
from catboost import CatBoostRegressor, Pool
from sklearn.decomposition import PCA
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score
from sklearn.model_selection import GroupKFold, KFold
from torchvision.models import ResNet18_Weights, resnet18

DB_PATH = Path('../instance/pricing_prediction.db') if Path('../instance/pricing_prediction.db').exists() else Path('instance/pricing_prediction.db')
RANDOM_STATE = 42
PRIMARY_METRIC = 'rmse'
SECONDARY_METRICS = ['mae', 'r2']
STRUCTURED_SPLITS = 5
IMAGE_SPLITS = 5
IMAGE_SAMPLE_PER_QUERY = 60
RUN_IMAGE_EXPERIMENT = True

pd.set_option('display.max_columns', 200)
pd.set_option('display.width', 160)
print(DB_PATH)


## Data loading

Load `product_snapshots`, enrich it with product metadata from `products`, and attach image aggregates from `product_images`.

A deliberate exception to plain `KFold` is used for the full structured model: the table contains repeated snapshots for some `sku_id`, so `GroupKFold` by `sku_id` avoids leakage between train and validation folds.


In [ ]:
FULL_SQL = '''
WITH image_agg AS (
    SELECT
        sku_id,
        COUNT(*) AS image_count,
        MAX(CASE WHEN position = 1 THEN image_url END) AS first_image_url
    FROM product_images
    GROUP BY sku_id
)
SELECT
    ps.id,
    ps.sku_id,
    ps.query,
    ps.page_number,
    ps.position,
    CAST(ps.current_price AS REAL) AS current_price,
    CAST(ps.original_price AS REAL) AS original_price,
    ps.discount_text,
    CAST(ps.rating AS REAL) AS rating,
    ps.review_count,
    COALESCE(NULLIF(ps.seller, ''), NULLIF(p.seller, ''), 'unknown') AS seller,
    CASE WHEN ps.sponsored THEN 1 ELSE 0 END AS sponsored,
    ps.scraped_at,
    p.product_id,
    COALESCE(NULLIF(p.brand, ''), 'unknown') AS brand,
    p.title,
    p.source_domain,
    img.image_count,
    img.first_image_url
FROM product_snapshots ps
JOIN products p ON p.sku_id = ps.sku_id
LEFT JOIN image_agg img ON img.sku_id = ps.sku_id
WHERE ps.current_price IS NOT NULL
'''

UNIQUE_IMAGE_SQL = '''
WITH image_agg AS (
    SELECT
        sku_id,
        COUNT(*) AS image_count,
        MAX(CASE WHEN position = 1 THEN image_url END) AS first_image_url
    FROM product_images
    GROUP BY sku_id
), ranked AS (
    SELECT
        ps.*,
        ROW_NUMBER() OVER (
            PARTITION BY ps.sku_id
            ORDER BY ps.scraped_at DESC, ps.page_number ASC, ps.position ASC
        ) AS rn
    FROM product_snapshots ps
    WHERE ps.current_price IS NOT NULL
)
SELECT
    r.sku_id,
    r.query,
    r.page_number,
    r.position,
    CAST(r.current_price AS REAL) AS current_price,
    CAST(r.original_price AS REAL) AS original_price,
    r.discount_text,
    CAST(r.rating AS REAL) AS rating,
    r.review_count,
    COALESCE(NULLIF(r.seller, ''), NULLIF(p.seller, ''), 'unknown') AS seller,
    CASE WHEN r.sponsored THEN 1 ELSE 0 END AS sponsored,
    COALESCE(NULLIF(p.brand, ''), 'unknown') AS brand,
    p.title,
    p.source_domain,
    img.image_count,
    img.first_image_url
FROM ranked r
JOIN products p ON p.sku_id = r.sku_id
LEFT JOIN image_agg img ON img.sku_id = r.sku_id
WHERE r.rn = 1 AND img.first_image_url IS NOT NULL
'''

with sqlite3.connect(DB_PATH) as conn:
    full_df = pd.read_sql_query(FULL_SQL, conn)
    unique_image_df = pd.read_sql_query(UNIQUE_IMAGE_SQL, conn)

summary = {
    'rows': len(full_df),
    'distinct_skus': full_df['sku_id'].nunique(),
    'distinct_queries': full_df['query'].nunique(),
    'priced_rows': int(full_df['current_price'].notna().sum()),
    'image_rows': int(unique_image_df['first_image_url'].notna().sum()),
}
summary


In [ ]:
def parse_discount(value: object) -> float:
    if value is None or value == '':
        return float('nan')
    match = re.search(r'(-?\d+(?:\.\d+)?)\s*%', str(value))
    if not match:
        return float('nan')
    return abs(float(match.group(1))) / 100.0


def query_root(value: str) -> str:
    text = str(value).lower()
    if 'zapatos' in text:
        return 'zapatos'
    if 'ropa' in text:
        return 'ropa'
    return 'other'


def query_audience(value: str) -> str:
    text = str(value).lower()
    if 'niñ' in text or 'nino' in text or 'niño' in text:
        return 'kids'
    if 'mujer' in text:
        return 'women'
    if 'hombre' in text:
        return 'men'
    return 'other'


def image_namespace(value: object) -> str:
    if not isinstance(value, str):
        return 'missing'
    parts = value.split('/')
    return parts[3] if len(parts) > 3 else 'missing'


def build_feature_frame(df: pd.DataFrame) -> pd.DataFrame:
    frame = df.copy()
    frame['discount_pct'] = frame['discount_text'].map(parse_discount)
    frame['has_discount'] = frame['discount_pct'].notna().astype(int)
    frame['discount_pct'] = frame['discount_pct'].fillna(-1.0)
    frame['original_price'] = frame['original_price'].fillna(-1.0)
    frame['has_original_price'] = (frame['original_price'] > 0).astype(int)
    frame['rating_missing'] = frame['rating'].isna().astype(int)
    frame['rating'] = frame['rating'].fillna(frame['rating'].median())
    frame['review_count'] = frame['review_count'].fillna(0)
    frame['review_count_log1p'] = np.log1p(frame['review_count'])
    frame['image_count'] = frame['image_count'].fillna(0).astype(int)
    frame['query_root'] = frame['query'].map(query_root)
    frame['query_audience'] = frame['query'].map(query_audience)
    frame['image_namespace'] = frame['first_image_url'].map(image_namespace)
    frame['title'] = frame['title'].fillna('')
    frame['title_word_count'] = frame['title'].str.split().str.len().fillna(0)
    frame['title_char_count'] = frame['title'].str.len().fillna(0)
    frame['title_digit_count'] = frame['title'].str.count(r'\d')
    frame['title_has_pack'] = frame['title'].str.contains(r'pack|set|kit', case=False, regex=True).astype(int)
    frame['title_has_kids_token'] = frame['title'].str.contains(r'niñ|nino|niño|kid|junior', case=False, regex=True).astype(int)
    frame['title_has_sport_token'] = frame['title'].str.contains(r'deportivo|running|trek|outdoor|gym|sport', case=False, regex=True).astype(int)
    frame['brand_in_title'] = frame.apply(lambda row: int(str(row['brand']).lower() in str(row['title']).lower()), axis=1)
    frame['rank_position'] = (frame['page_number'] - 1) * 48 + frame['position']
    frame['log_target'] = np.log1p(frame['current_price'])
    return frame

full_features_df = build_feature_frame(full_df)
unique_image_features_df = build_feature_frame(unique_image_df)
full_features_df.head(3)


## Structured CatBoost experiments

These are the main price-prediction experiments on the full dataset. The strongest model uses listing signals such as `original_price` and `discount_pct`, along with ranking, seller/brand/query context, review features, and title-derived features.


In [ ]:
STRUCTURED_EXPERIMENTS = {
    'cb_core_structured': {
        'family': 'catboost',
        'features': [
            'query', 'query_root', 'query_audience', 'brand', 'seller', 'source_domain', 'image_namespace',
            'page_number', 'position', 'rank_position', 'rating', 'rating_missing', 'review_count_log1p',
            'image_count', 'title_word_count', 'title_char_count', 'title_digit_count', 'title_has_pack',
            'title_has_kids_token', 'title_has_sport_token', 'brand_in_title', 'sponsored'
        ],
        'cat_features': ['query', 'query_root', 'query_audience', 'brand', 'seller', 'source_domain', 'image_namespace'],
        'params': {'depth': 7, 'learning_rate': 0.06, 'iterations': 300},
        'feature_blocks': ['query/category', 'brand-seller', 'ranking', 'rating-review', 'title-derived', 'image metadata'],
    },
    'cb_core_plus_listing': {
        'family': 'catboost',
        'features': [
            'query', 'query_root', 'query_audience', 'brand', 'seller', 'source_domain', 'image_namespace',
            'page_number', 'position', 'rank_position', 'rating', 'rating_missing', 'review_count_log1p',
            'image_count', 'title_word_count', 'title_char_count', 'title_digit_count', 'title_has_pack',
            'title_has_kids_token', 'title_has_sport_token', 'brand_in_title', 'sponsored',
            'original_price', 'discount_pct', 'has_original_price', 'has_discount'
        ],
        'cat_features': ['query', 'query_root', 'query_audience', 'brand', 'seller', 'source_domain', 'image_namespace'],
        'params': {'depth': 7, 'learning_rate': 0.06, 'iterations': 350},
        'feature_blocks': ['query/category', 'brand-seller', 'ranking', 'rating-review', 'title-derived', 'image metadata', 'listing signals'],
    },
    'cb_listing_plus_price_position': {
        'family': 'catboost',
        'features': [
            'query', 'query_root', 'query_audience', 'brand', 'seller', 'source_domain', 'image_namespace',
            'page_number', 'position', 'rank_position', 'rating', 'rating_missing', 'review_count_log1p',
            'image_count', 'title_word_count', 'title_char_count', 'title_digit_count', 'title_has_pack',
            'title_has_kids_token', 'title_has_sport_token', 'brand_in_title', 'sponsored',
            'original_price', 'discount_pct', 'has_original_price', 'has_discount'
        ],
        'cat_features': ['query', 'query_root', 'query_audience', 'brand', 'seller', 'source_domain', 'image_namespace'],
        'params': {'depth': 8, 'learning_rate': 0.05, 'iterations': 450},
        'feature_blocks': ['query/category', 'brand-seller', 'ranking', 'rating-review', 'title-derived', 'image metadata', 'listing signals'],
    },
}


def evaluate_grouped_catboost(frame: pd.DataFrame, experiments: dict[str, dict]) -> pd.DataFrame:
    splitter = GroupKFold(n_splits=STRUCTURED_SPLITS)
    rows = []
    for name, cfg in experiments.items():
        fold_rmse = []
        fold_mae = []
        fold_r2 = []
        for fold, (train_idx, valid_idx) in enumerate(splitter.split(frame, groups=frame['sku_id']), start=1):
            train_df = frame.iloc[train_idx]
            valid_df = frame.iloc[valid_idx]
            train_pool = Pool(train_df[cfg['features']], label=train_df['log_target'], cat_features=cfg['cat_features'])
            valid_pool = Pool(valid_df[cfg['features']], label=valid_df['log_target'], cat_features=cfg['cat_features'])
            model = CatBoostRegressor(
                loss_function='RMSE',
                eval_metric='RMSE',
                random_seed=RANDOM_STATE,
                verbose=False,
                allow_writing_files=False,
                **cfg['params'],
            )
            model.fit(train_pool, eval_set=valid_pool, use_best_model=True)
            pred = np.expm1(model.predict(valid_pool))
            actual = np.expm1(valid_df['log_target'])
            fold_rmse.append(math.sqrt(mean_squared_error(actual, pred)))
            fold_mae.append(mean_absolute_error(actual, pred))
            fold_r2.append(r2_score(actual, pred))
        rows.append({
            'name': name,
            'family': cfg['family'],
            'cv_mean_rmse': float(np.mean(fold_rmse)),
            'cv_std_rmse': float(np.std(fold_rmse)),
            'cv_mean_mae': float(np.mean(fold_mae)),
            'cv_std_mae': float(np.std(fold_mae)),
            'cv_mean_r2': float(np.mean(fold_r2)),
            'cv_std_r2': float(np.std(fold_r2)),
            'fold_rmse': [round(float(x), 4) for x in fold_rmse],
            'feature_blocks': cfg['feature_blocks'],
            'params': cfg['params'],
        })
    leaderboard = pd.DataFrame(rows).sort_values(['cv_mean_rmse', 'cv_std_rmse']).reset_index(drop=True)
    return leaderboard

structured_leaderboard = evaluate_grouped_catboost(full_features_df, STRUCTURED_EXPERIMENTS)
structured_leaderboard


## Optional PyTorch image experiment

This experiment uses the first product image, extracts ResNet18 embeddings with PyTorch, and compresses them with PCA before passing them into CatBoost on a sampled unique-SKU subset.

The first recorded run used `mps` successfully.


In [ ]:
def build_image_sample(frame: pd.DataFrame, sample_per_query: int = IMAGE_SAMPLE_PER_QUERY) -> pd.DataFrame:
    parts = []
    for _, group in frame.groupby('query'):
        parts.append(group.sample(n=min(len(group), sample_per_query), random_state=RANDOM_STATE))
    return pd.concat(parts, ignore_index=True)


def fetch_first_image(url: str, client: httpx.Client) -> Image.Image | None:
    try:
        response = client.get(url)
        response.raise_for_status()
        return Image.open(io.BytesIO(response.content)).convert('RGB')
    except Exception:
        return None


def compute_image_embeddings(frame: pd.DataFrame) -> tuple[pd.DataFrame, list[str], str]:
    weights = ResNet18_Weights.DEFAULT
    preprocess = weights.transforms()
    device = torch.device('mps' if torch.backends.mps.is_available() else 'cpu')
    backbone = resnet18(weights=weights)
    backbone.fc = torch.nn.Identity()
    backbone.eval()
    backbone.to(device)

    client = httpx.Client(timeout=20.0, follow_redirects=True, headers={'User-Agent': 'pricing-prediction-experiment/1.0'})
    images: dict[int, Image.Image] = {}
    with ThreadPoolExecutor(max_workers=12) as pool:
        future_map = {pool.submit(fetch_first_image, row.first_image_url, client): idx for idx, row in frame.iterrows()}
        for future in as_completed(future_map):
            idx = future_map[future]
            image = future.result()
            if image is not None:
                images[idx] = image
    client.close()

    valid_indices = sorted(images)
    valid_frame = frame.iloc[valid_indices].reset_index(drop=True)
    batch_tensors = [preprocess(images[idx]) for idx in valid_indices]
    stack = torch.stack(batch_tensors)
    embeddings = []
    with torch.inference_mode():
        for start in range(0, len(stack), 32):
            batch = stack[start:start + 32].to(device)
            embeddings.append(backbone(batch).detach().cpu().numpy())
    embedding_matrix = np.vstack(embeddings)
    pca = PCA(n_components=min(24, embedding_matrix.shape[0], embedding_matrix.shape[1]), random_state=RANDOM_STATE)
    image_pcs = pca.fit_transform(embedding_matrix)
    image_cols = []
    for idx in range(image_pcs.shape[1]):
        col = f'image_pc_{idx + 1}'
        valid_frame[col] = image_pcs[:, idx]
        image_cols.append(col)
    return valid_frame, image_cols, str(device)


def evaluate_image_hybrid(frame: pd.DataFrame, image_cols: list[str]) -> pd.DataFrame:
    base_features = [
        'query', 'query_root', 'query_audience', 'brand', 'seller', 'source_domain',
        'page_number', 'position', 'rank_position', 'rating', 'rating_missing', 'review_count_log1p',
        'image_count', 'title_word_count', 'title_char_count', 'title_digit_count', 'title_has_pack',
        'title_has_kids_token', 'title_has_sport_token', 'brand_in_title', 'sponsored'
    ]
    cat_features = ['query', 'query_root', 'query_audience', 'brand', 'seller', 'source_domain']
    experiments = {
        'cb_subset_structured': base_features,
        'cb_subset_structured_plus_image': base_features + image_cols,
    }

    rows = []
    splitter = KFold(n_splits=IMAGE_SPLITS, shuffle=True, random_state=RANDOM_STATE)
    for name, features in experiments.items():
        fold_rmse = []
        fold_mae = []
        fold_r2 = []
        for train_idx, valid_idx in splitter.split(frame):
            train_df = frame.iloc[train_idx]
            valid_df = frame.iloc[valid_idx]
            train_pool = Pool(train_df[features], label=train_df['log_target'], cat_features=[c for c in cat_features if c in features])
            valid_pool = Pool(valid_df[features], label=valid_df['log_target'], cat_features=[c for c in cat_features if c in features])
            model = CatBoostRegressor(
                loss_function='RMSE',
                eval_metric='RMSE',
                random_seed=RANDOM_STATE,
                verbose=False,
                allow_writing_files=False,
                iterations=260,
                depth=6,
                learning_rate=0.05,
            )
            model.fit(train_pool, eval_set=valid_pool, use_best_model=True)
            pred = np.expm1(model.predict(valid_pool))
            actual = np.expm1(valid_df['log_target'])
            fold_rmse.append(math.sqrt(mean_squared_error(actual, pred)))
            fold_mae.append(mean_absolute_error(actual, pred))
            fold_r2.append(r2_score(actual, pred))
        rows.append({
            'name': name,
            'family': 'catboost',
            'cv_mean_rmse': float(np.mean(fold_rmse)),
            'cv_std_rmse': float(np.std(fold_rmse)),
            'cv_mean_mae': float(np.mean(fold_mae)),
            'cv_std_mae': float(np.std(fold_mae)),
            'cv_mean_r2': float(np.mean(fold_r2)),
            'cv_std_r2': float(np.std(fold_r2)),
            'fold_rmse': [round(float(x), 4) for x in fold_rmse],
        })
    return pd.DataFrame(rows).sort_values(['cv_mean_rmse', 'cv_std_rmse']).reset_index(drop=True)

if RUN_IMAGE_EXPERIMENT:
    image_sample = build_image_sample(unique_image_features_df)
    image_frame, image_cols, image_device = compute_image_embeddings(image_sample)
    image_leaderboard = evaluate_image_hybrid(image_frame, image_cols)
    print({'image_sample_rows': len(image_sample), 'downloaded_image_rows': len(image_frame), 'device': image_device})
    image_leaderboard
else:
    print('Set RUN_IMAGE_EXPERIMENT = True to rerun the PyTorch image experiment.')


## Recorded experiment registry

Use this as a fixed checkpoint from the initial run before starting new iterations.


In [ ]:
recorded_results = pd.DataFrame([
    {
        'name': 'cb_core_structured',
        'family': 'catboost',
        'scope': 'full_dataset',
        'cv_mean_rmse': 64.8424,
        'cv_std_rmse': 5.0117,
        'cv_mean_mae': 33.7213,
        'cv_std_mae': 0.9938,
        'cv_mean_r2': 0.6930,
        'cv_std_r2': 0.0356,
        'fold_rmse': [60.2747, 58.6628, 64.6263, 72.1115, 68.5368],
        'feature_blocks': ['query/category', 'brand-seller', 'ranking', 'rating-review', 'title-derived', 'image metadata'],
        'params': {'depth': 7, 'learning_rate': 0.06, 'iterations': 300},
        'notes': 'Content-only structured CatBoost baseline.',
    },
    {
        'name': 'cb_core_plus_listing',
        'family': 'catboost',
        'scope': 'full_dataset',
        'cv_mean_rmse': 48.2305,
        'cv_std_rmse': 3.7295,
        'cv_mean_mae': 18.8418,
        'cv_std_mae': 0.3255,
        'cv_mean_r2': 0.8300,
        'cv_std_r2': 0.0212,
        'fold_rmse': [44.5376, 45.3585, 45.8892, 53.8055, 51.5615],
        'feature_blocks': ['query/category', 'brand-seller', 'ranking', 'rating-review', 'title-derived', 'image metadata', 'listing signals'],
        'params': {'depth': 7, 'learning_rate': 0.06, 'iterations': 350},
        'notes': 'Adding listing signals brought a large lift.',
    },
    {
        'name': 'cb_listing_plus_price_position',
        'family': 'catboost',
        'scope': 'full_dataset',
        'cv_mean_rmse': 45.8882,
        'cv_std_rmse': 3.7036,
        'cv_mean_mae': 17.7949,
        'cv_std_mae': 0.2750,
        'cv_mean_r2': 0.8460,
        'cv_std_r2': 0.0201,
        'fold_rmse': [41.8493, 43.1609, 43.8732, 51.3321, 49.2254],
        'feature_blocks': ['query/category', 'brand-seller', 'ranking', 'rating-review', 'title-derived', 'image metadata', 'listing signals'],
        'params': {'depth': 8, 'learning_rate': 0.05, 'iterations': 450},
        'notes': 'Winner on the full dataset.',
    },
    {
        'name': 'cb_subset_structured',
        'family': 'catboost',
        'scope': 'image_sample_360',
        'cv_mean_rmse': 79.3127,
        'cv_std_rmse': 22.1919,
        'cv_mean_mae': 48.1199,
        'cv_std_mae': 9.3683,
        'cv_mean_r2': 0.4317,
        'cv_std_r2': 0.1402,
        'fold_rmse': [88.5471, 52.2984, 61.7546, 78.0782, 115.8852],
        'feature_blocks': ['query/category', 'brand-seller', 'ranking', 'rating-review', 'title-derived'],
        'params': {'depth': 6, 'learning_rate': 0.05, 'iterations': 260},
        'notes': 'Structured subset baseline for the image sample.',
    },
    {
        'name': 'cb_subset_structured_plus_image',
        'family': 'catboost + pytorch',
        'scope': 'image_sample_360',
        'cv_mean_rmse': 78.9462,
        'cv_std_rmse': 20.4869,
        'cv_mean_mae': 49.4321,
        'cv_std_mae': 8.3775,
        'cv_mean_r2': 0.4379,
        'cv_std_r2': 0.1075,
        'fold_rmse': [84.6919, 52.1685, 64.9664, 80.0180, 112.8861],
        'feature_blocks': ['query/category', 'brand-seller', 'ranking', 'rating-review', 'title-derived', 'resnet18 image embeddings (pca)'],
        'params': {'depth': 6, 'learning_rate': 0.05, 'iterations': 260, 'device': 'mps'},
        'notes': 'Image embeddings gave a marginal RMSE lift but worsened MAE, so this is not the champion yet.',
    },
]).sort_values(['cv_mean_rmse', 'cv_std_rmse']).reset_index(drop=True)

recorded_results


## Next iteration ideas

- Add a text-aware CatBoost experiment with `title` as a true text feature instead of only derived counts.
- Build query-specific models for `ropa` and `zapatos` and compare them against the global model.
- Expand the image experiment from 360 SKUs to a larger unique-SKU sample and test stronger backbones or better fusion strategies.
- Compare predicting `log1p(current_price)` versus raw `current_price` for the image hybrid branch.
- Test a leakage-resistant version that excludes `original_price` and `discount_pct` if the production use case will not know those fields at prediction time.
